# Phase I — SAM mask repair (post-processing)

**No retraining.** Repairs existing Phase I masks with [SAM](https://arxiv.org/abs/2304.02643).

1. Mount Drive
2. Install packages
3. Run the **single repair cell** below


## Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Install packages


In [ ]:
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "transformers", "accelerate", "opencv-python", "matplotlib", "tqdm",
], check=True)
print('ok')


## Run SAM repair (all-in-one)

Edit `DRIVE_BASE` / `INPUT_MASK_DIR` at the top of the cell if needed.


In [ ]:
# ============ CONFIG (edit if needed) ============
import os

DRIVE_BASE = "/content/drive/MyDrive/RouteResilience"
DATA_DIR = f"{DRIVE_BASE}/datasets/train"
INPUT_MASK_DIR = f"{DRIVE_BASE}/outputs/masks_deeplab"
OUTPUT_MASK_DIR = f"{DRIVE_BASE}/outputs/masks_sam_repaired"
SAM_MODEL_ID = "facebook/sam-vit-base"
DEMO_IDS = ["493626", "477671", "422265", "194764"]
RUN_ALL = False
# ================================================

os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)
if not os.path.isdir(INPUT_MASK_DIR):
    raise FileNotFoundError(f"INPUT_MASK_DIR not found: {INPUT_MASK_DIR}")

import cv2
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import SamModel, SamProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

processor = SamProcessor.from_pretrained(SAM_MODEL_ID)
sam = SamModel.from_pretrained(SAM_MODEL_ID).to(device).eval()
print("Loaded", SAM_MODEL_ID)
print("Input masks :", INPUT_MASK_DIR)
print("Output masks:", OUTPUT_MASK_DIR)


def sample_points(xs, ys, n=12, seed=0):
    if len(xs) == 0:
        return []
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(xs), size=min(n, len(xs)), replace=False)
    return [[int(xs[i]), int(ys[i])] for i in idx]


def gap_bridge_points(mask_bin, max_pairs=6):
    n_labels, labels = cv2.connectedComponents(mask_bin.astype(np.uint8))
    if n_labels <= 2:
        return []
    comps = []
    for lab in range(1, n_labels):
        ys, xs = np.where(labels == lab)
        if len(xs) < 50:
            continue
        comps.append((xs.mean(), ys.mean(), len(xs)))
    comps.sort(key=lambda x: x[2], reverse=True)
    comps = comps[:8]
    points = []
    for i in range(len(comps)):
        for j in range(i + 1, len(comps)):
            x1, y1 = comps[i][0], comps[i][1]
            x2, y2 = comps[j][0], comps[j][1]
            if np.hypot(x1 - x2, y1 - y2) > mask_bin.shape[0] * 0.45:
                continue
            points.append([int((x1 + x2) / 2), int((y1 + y2) / 2)])
            if len(points) >= max_pairs:
                return points
    return points


def connectivity_stats(mask_bin):
    n_labels, labels = cv2.connectedComponents(mask_bin.astype(np.uint8))
    areas = [(labels == i).sum() for i in range(1, n_labels)]
    total = mask_bin.sum()
    largest = max(areas) if areas else 0
    ratio = float(largest / total) if total > 0 else 0.0
    return n_labels - 1, ratio


def morph_fallback(mask_bin, k_close):
    return cv2.morphologyEx(mask_bin.astype(np.uint8), cv2.MORPH_CLOSE, k_close)


@torch.no_grad()
def repair_mask_with_sam(image_rgb, coarse_mask, seed=42):
    h, w = image_rgb.shape[:2]
    if coarse_mask.shape[:2] != (h, w):
        coarse_mask = cv2.resize(coarse_mask, (w, h), interpolation=cv2.INTER_NEAREST)
    mask_bin = (coarse_mask > 127).astype(np.uint8)
    k_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))

    if mask_bin.sum() == 0:
        return coarse_mask, np.zeros_like(mask_bin)

    road_core = cv2.erode(mask_bin, k_small, iterations=1)
    ys, xs = np.where(road_core > 0)
    if len(xs) < 5:
        ys, xs = np.where(mask_bin > 0)

    pos = sample_points(xs, ys, n=14, seed=seed)
    gap_pts = gap_bridge_points(mask_bin)

    dilated = cv2.dilate(mask_bin, k_close, iterations=2)
    bg_ys, bg_xs = np.where(dilated == 0)
    neg = sample_points(bg_xs, bg_ys, n=8, seed=seed + 1)

    if len(pos) == 0 and len(gap_pts) == 0:
        repaired = morph_fallback(mask_bin, k_close)
        return (repaired * 255).astype(np.uint8), np.zeros_like(mask_bin)

    all_pts = pos + gap_pts + neg
    all_labels = [1] * len(pos) + [1] * len(gap_pts) + [0] * len(neg)

    pil_image = Image.fromarray(image_rgb)
    inputs = processor(
        images=pil_image,
        input_points=[all_pts],
        input_labels=[all_labels],
        return_tensors="pt",
    )
    inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}

    out = sam(**inputs)
    masks = processor.image_processor.post_process_masks(
        out.pred_masks.detach().cpu(),
        inputs["original_sizes"].detach().cpu(),
        inputs["reshaped_input_sizes"].detach().cpu(),
    )[0]

    scores = out.iou_scores[0].detach().cpu().numpy().reshape(-1)
    if masks.ndim == 4:
        masks = masks[0]
    best_idx = int(scores.argmax())
    sam_mask = masks[best_idx].bool().cpu().numpy().astype(np.uint8)
    if sam_mask.shape != mask_bin.shape:
        sam_mask = cv2.resize(sam_mask, (w, h), interpolation=cv2.INTER_NEAREST)

    touch = cv2.dilate(mask_bin, k_close, iterations=1)
    sam_added = ((sam_mask & touch) | mask_bin).astype(np.uint8)
    repaired = morph_fallback(sam_added, k_close)
    return (repaired * 255).astype(np.uint8), (sam_mask * 255).astype(np.uint8)


# --- run repair ---
if RUN_ALL:
    tile_ids = [f.replace("_pred.png", "") for f in sorted(os.listdir(INPUT_MASK_DIR)) if f.endswith("_pred.png")]
else:
    tile_ids = DEMO_IDS

results = []
for img_id in tqdm(tile_ids, desc="SAM repair"):
    sat_path = f"{DATA_DIR}/{img_id}_sat.jpg"
    in_path = f"{INPUT_MASK_DIR}/{img_id}_pred.png"
    out_path = f"{OUTPUT_MASK_DIR}/{img_id}_pred.png"

    if not os.path.exists(sat_path):
        print("skip (no sat):", img_id)
        continue
    if not os.path.exists(in_path):
        print("skip (no mask):", img_id)
        continue

    try:
        image = cv2.cvtColor(cv2.imread(sat_path), cv2.COLOR_BGR2RGB)
        coarse = cv2.imread(in_path, cv2.IMREAD_GRAYSCALE)
        repaired, _ = repair_mask_with_sam(image, coarse, seed=int(img_id) % 10000)
        cv2.imwrite(out_path, repaired)

        before_bin = (coarse > 127).astype(np.uint8)
        after_bin = (repaired > 127).astype(np.uint8)
        n_before, r_before = connectivity_stats(before_bin)
        n_after, r_after = connectivity_stats(after_bin)
        results.append({
            "id": img_id,
            "components_before": n_before,
            "components_after": n_after,
            "largest_frac_before": r_before,
            "largest_frac_after": r_after,
            "pixels_added": int(after_bin.sum() - before_bin.sum()),
        })
        print(
            f"{img_id} | components {n_before}->{n_after} | "
            f"largest-frac {r_before:.3f}->{r_after:.3f} | +pixels {results[-1]['pixels_added']}"
        )
    except Exception as e:
        print(f"ERROR on {img_id}: {type(e).__name__}: {e}")

if not results:
    raise RuntimeError("No tiles repaired. Check INPUT_MASK_DIR and DATA_DIR.")

print("Saved repaired masks to", OUTPUT_MASK_DIR)

# --- before / after plot ---
fig, axes = plt.subplots(len(results), 4, figsize=(16, 4 * len(results)))
if len(results) == 1:
    axes = axes.reshape(1, -1)

for row, r in enumerate(results):
    img_id = r["id"]
    sat = cv2.cvtColor(cv2.imread(f"{DATA_DIR}/{img_id}_sat.jpg"), cv2.COLOR_BGR2RGB)
    before = cv2.imread(f"{INPUT_MASK_DIR}/{img_id}_pred.png", cv2.IMREAD_GRAYSCALE)
    after = cv2.imread(f"{OUTPUT_MASK_DIR}/{img_id}_pred.png", cv2.IMREAD_GRAYSCALE)
    h, w = sat.shape[:2]
    if before.shape[:2] != (h, w):
        before = cv2.resize(before, (w, h), interpolation=cv2.INTER_NEAREST)

    axes[row, 0].imshow(sat); axes[row, 0].set_title(f"{img_id} satellite"); axes[row, 0].axis("off")
    axes[row, 1].imshow(before, cmap="gray"); axes[row, 1].set_title("Before"); axes[row, 1].axis("off")
    axes[row, 2].imshow(after, cmap="gray"); axes[row, 2].set_title("SAM repaired"); axes[row, 2].axis("off")
    diff = ((after > 127).astype(np.uint8) - (before > 127).astype(np.uint8)).clip(0, 1)
    axes[row, 3].imshow(diff, cmap="hot"); axes[row, 3].set_title("Added pixels"); axes[row, 3].axis("off")

plt.tight_layout()
plt.savefig(f"{OUTPUT_MASK_DIR}/sam_repair_comparison.png", dpi=120)
plt.show()
print("Phase II: set MASK_DIR to masks_sam_repaired or copy into masks_deeplab/")
